# Sentiment Analysis - Cross Validation
Dataset: Restaurants_Train_v2.csv (SemEval-14 ABSA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn import neighbors
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate, learning_curve
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    make_scorer, f1_score
)
from sklearn.pipeline import Pipeline

print('Libraries loaded OK')

## 1. Load & Prepare Data

In [ ]:
DATA_PATH = '/home/conanwinner/Desktop/_CODE/VKU_Lab_Sentiment/Restaurants_Train_v2.csv'

df = pd.read_csv(DATA_PATH, encoding='utf8')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
# Loại bỏ nhãn 'conflict' nếu có (ít mẫu, gây nhiễu)
df = df[df['polarity'] != 'conflict'].copy()

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
label_names = ['negative', 'neutral', 'positive']

df['labels'] = df['polarity'].map(label_map)
df = df.dropna(subset=['labels', 'Sentence'])
df['labels'] = df['labels'].astype(int)

print('Label distribution:')
print(df['polarity'].value_counts())
print('Total samples:', len(df))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['polarity'].value_counts()
bars = ax.bar(counts.index, counts.values, color=['#e74c3c','#f39c12','#2ecc71'], alpha=0.85)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11)
ax.set_title('Phân bổ nhãn cảm xúc', fontsize=13)
ax.set_ylabel('Số mẫu')
plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
nlp = spacy.load('en_core_web_sm')

KEEP_WORDS = {
    'not','no','never','none','nobody','nothing','neither','nor',
    'cannot','without',"n't",'ca','noone',
    'very','too','so','quite','really','most','least','less','much',
    'enough','almost','pretty',
    'but','however','although','though','nevertheless','yet','whereas',
    'always','often','sometimes','ever'
}

stop_words = nlp.Defaults.stop_words
sw = {w for w in stop_words if w not in KEEP_WORDS}

def preprocess(text):
    text = text.lower()
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if not token.is_punct and token.text not in sw
    ]
    return ' '.join(tokens)

print('Preprocessing...')
df['text_clean'] = df['Sentence'].apply(preprocess)
print('Done!')
df[['Sentence','text_clean']].head(3)

## 3. Cross-Validation Setup

In [ ]:
X = df['text_clean'].values
y = df['labels'].values

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'f1_macro': make_scorer(f1_score, average='macro'),
    'f1_weighted': make_scorer(f1_score, average='weighted'),
}

def myweight(distances):
    sigma2 = 0.3
    return np.exp(-distances**2 / sigma2)

# Định nghĩa các pipeline (TF-IDF + model)
models = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', MultinomialNB())
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'SVM': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', SVC(kernel='linear', decision_function_shape='ovo'))
    ]),
    'KNN': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', neighbors.KNeighborsClassifier(n_neighbors=7, p=2, weights=myweight))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
}

print(f'Models: {list(models.keys())}')
print(f'CV folds: {N_SPLITS}')

## 4. Run Cross-Validation

In [ ]:
cv_results = {}

for name, pipeline in models.items():
    print(f'Running CV: {name}...')
    result = cross_validate(
        pipeline, X, y,
        cv=skf,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )
    cv_results[name] = result
    acc_mean = result['test_accuracy'].mean()
    acc_std  = result['test_accuracy'].std()
    f1_mean  = result['test_f1_weighted'].mean()
    print(f'  Accuracy: {acc_mean:.4f} ± {acc_std:.4f} | F1-weighted: {f1_mean:.4f}')

print('\nDone!')

## 5. Visualize CV Results

In [ ]:
# --- Bảng tổng hợp ---
summary = []
for name, res in cv_results.items():
    summary.append({
        'Model': name,
        'Train Acc': res['train_accuracy'].mean(),
        'Test Acc':  res['test_accuracy'].mean(),
        'Test Acc Std': res['test_accuracy'].std(),
        'F1 Macro':  res['test_f1_macro'].mean(),
        'F1 Weighted': res['test_f1_weighted'].mean(),
    })

df_summary = pd.DataFrame(summary).set_index('Model')
print(df_summary.round(4).to_string())

In [ ]:
# --- Biểu đồ so sánh Accuracy (mean ± std) ---
model_names = list(cv_results.keys())
test_means  = [cv_results[m]['test_accuracy'].mean() for m in model_names]
test_stds   = [cv_results[m]['test_accuracy'].std()  for m in model_names]
train_means = [cv_results[m]['train_accuracy'].mean() for m in model_names]

x = np.arange(len(model_names))
width = 0.35
colors_train = '#3498db'
colors_test  = '#e74c3c'

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, train_means, width, label='Train Accuracy', color=colors_train, alpha=0.8)
bars2 = ax.bar(x + width/2, test_means,  width, label='CV Test Accuracy', color=colors_test,  alpha=0.8,
               yerr=test_stds, capsize=5, error_kw={'elinewidth':2})

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar, std in zip(bars2, test_stds):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title(f'Train vs CV Test Accuracy ({N_SPLITS}-Fold Stratified CV)', fontsize=12)
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Biểu đồ accuracy từng fold (line plot) ---
fig, ax = plt.subplots(figsize=(10, 5))
fold_labels = [f'Fold {i+1}' for i in range(N_SPLITS)]
colors_line = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']

for (name, res), color in zip(cv_results.items(), colors_line):
    ax.plot(fold_labels, res['test_accuracy'], marker='o', label=name, color=color, linewidth=2)

ax.set_ylabel('Accuracy')
ax.set_title('CV Accuracy per Fold', fontsize=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# --- Biểu đồ so sánh Accuracy / F1-macro / F1-weighted ---
metrics_to_plot = ['Test Acc', 'F1 Macro', 'F1 Weighted']
x = np.arange(len(metrics_to_plot))
width = 0.15
colors_bar = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, color) in enumerate(zip(model_names, colors_bar)):
    vals = [
        cv_results[name]['test_accuracy'].mean(),
        cv_results[name]['test_f1_macro'].mean(),
        cv_results[name]['test_f1_weighted'].mean(),
    ]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width*(len(model_names)-1)/2)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: CV Metrics', fontsize=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 6. Learning Curves (Train size vs Score)

In [ ]:
train_sizes_pct = np.linspace(0.1, 1.0, 8)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (name, pipeline) in zip(axes, models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        pipeline, X, y,
        cv=skf,
        train_sizes=train_sizes_pct,
        scoring='accuracy',
        n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'o-', color='#3498db', label='Train', linewidth=2)
    ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='#3498db')
    ax.plot(train_sizes, val_mean, 's-', color='#e74c3c', label='CV Val', linewidth=2)
    ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.15, color='#e74c3c')

    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Training samples')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

# Ẩn subplot thừa
for ax in axes[len(models):]:
    ax.set_visible(False)

fig.suptitle('Learning Curves (Accuracy vs Training Size)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Final Evaluation (Train full → Test split)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

final_results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True)
    final_results[name] = {'y_pred': y_pred, 'report': report}
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=label_names))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (name, res) in zip(axes, final_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

for ax in axes[len(final_results):]:
    ax.set_visible(False)

plt.suptitle('Confusion Matrices (Hold-out Test Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Bảng tổng hợp cuối cùng: CV vs Hold-out
rows = []
for name in models:
    rows.append({
        'Model': name,
        'CV Acc (mean)': round(cv_results[name]['test_accuracy'].mean(), 4),
        'CV Acc (std)':  round(cv_results[name]['test_accuracy'].std(),  4),
        'CV F1-W':       round(cv_results[name]['test_f1_weighted'].mean(), 4),
        'Holdout Acc':   round(final_results[name]['report']['accuracy'], 4),
        'Holdout F1-W':  round(final_results[name]['report']['weighted avg']['f1-score'], 4),
    })

df_final = pd.DataFrame(rows).set_index('Model')
print(df_final.to_string())
df_final